In [16]:
from seeq import spy
spy.jobs.schedule("every day at 8:00 AM")

[spy.jobs MOCK] schedule('every day at 8:00 AM') — ignorado localmente


# 00 · Geração do hour_prev.csv

Notebook-raiz do pipeline. Puxa os sinais do Seeq, aplica pré-processamento e
calcula `target_rul` e as features de degradação usadas pelos notebooks 03–07.

**Saída:** `hour_prev.csv` com as colunas:

| Coluna | Origem |
|---|---|
| `Timestamp` | Seeq (índice temporal) |
| `Forca_A`, `Forca_B` | Sinais de força (Seeq) |
| `Media` | (Forca_A + Forca_B) / 2 |
| `Delta_AB` | \|Forca_A − Forca_B\| |
| `target_rul` | Horas até a próxima troca confirmada (de `troca_modulo copy.csv`) |
| `slope_Media_14d` | Inclinação rolling 14d da Média |
| `std_Delta_AB_7d` | Desvio padrão rolling 7d do Delta_AB |
| `media_ratio_14d` | Razão Media / média_móvel_14d |

> **Execute este notebook antes de qualquer outro.** Notebooks 03, 05, 06 e 07
> dependem do `hour_prev.csv` gerado aqui.

## 1. Configuração

In [17]:
import sys
sys.path.insert(0, '..')

import warnings
import numpy as np
import pandas as pd
from seeq import spy
from src.predictor import build_rul_target, extract_degradation_features

# ── Parâmetros ────────────────────────────────────────────────────────────────
WORKBOOK_ID       = '0F1373E4-EE13-EC80-9487-F11E5446FD54'
ANOS_HISTORICO    = 4          # janela de pull
IQR_MULTIPLIER    = 1.0        # fator do filtro de Delta_AB
ARQUIVO_TROCAS    = 'troca_modulo.csv'   # todas as trocas confirmadas
COLUNA_DATA_TROCA = 'Data-base do inicio'
OUTPUT_CSV        = '00_hour_prev.csv'

# IDs dos sinais no workbook
SIGNAL_IDS = {
    'Forca_A'   : '951157FA-D4BB-4696-A534-AEE4B48532CB',
    'Forca_B'   : '8D9E2FE1-6000-438C-B293-0EDDAA182851'
}
# ─────────────────────────────────────────────────────────────────────────────

print('Configuração OK')
print(f'  Workbook : {WORKBOOK_ID}')
print(f'  Histórico: {ANOS_HISTORICO} anos')
print(f'  Filtro   : IQR × {IQR_MULTIPLIER}')

Configuração OK
  Workbook : 0F1373E4-EE13-EC80-9487-F11E5446FD54
  Histórico: 4 anos
  Filtro   : IQR × 1.0


## 2. Pull do Seeq

In [18]:
user_tz   = spy.utils.get_user_timezone(spy.session)
end_time  = pd.Timestamp.now(tz=user_tz)
start_time = end_time - pd.DateOffset(years=ANOS_HISTORICO)

items_df = pd.DataFrame([
    {'ID': sid, 'Type': 'Signal'}
    for sid in SIGNAL_IDS.values()
])

print(f'Período: {start_time.date()} → {end_time.date()}')
print('Puxando dados do Seeq...')

data_raw = spy.pull(
    items_df,
    start=start_time.isoformat(),
    end=end_time.isoformat(),
    grid=None,
    header='ID',
)

# Renomear IDs → nomes legíveis
id_to_name = {v: k for k, v in SIGNAL_IDS.items()}
data_raw = data_raw.rename(columns=id_to_name)

print(f'Dados recebidos: {len(data_raw):,} leituras')
data_raw.head()

Período: 2022-05-05 → 2026-05-05
Puxando dados do Seeq...
[spy MOCK] pull() — lendo dados locais (sem Seeq)
           Sinais: 2 | Período: 2022-05-05T09:51:56.831375-03:00 → 2026-05-05T09:51:56.831375-03:00
           CSV: 2,429 linhas de '00_hour_prev.csv'
Dados recebidos: 2,429 leituras


,Forca_A,Forca_B,Media,Delta_AB,slope_Media_14d,std_Delta_AB_7d,media_ratio_14d
Timestamp,,,,,,,
2022-06-27 15:16:07-03:00,712.2,689.40,700.800,22.80,0.000000,0.000000,1.000000
2022-07-04 03:08:01-03:00,608.9,563.50,586.200,45.40,-17.646040,15.980613,0.910956
2022-07-05 03:03:01-03:00,742.9,730.75,736.825,12.15,-3.266336,23.511300,1.092226
2022-07-11 03:09:40-03:00,752.1,708.20,730.150,43.90,2.836505,22.450640,1.060503
2022-07-12 19:49:01-03:00,375.6,350.50,363.050,25.10,-19.310718,13.293607,0.601020


## 3. Pré-processamento

In [19]:
df = data_raw.copy()

# Timestamp como coluna
if isinstance(df.index, pd.DatetimeIndex):
    df = df.reset_index().rename(columns={'index': 'Timestamp'})
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values('Timestamp').reset_index(drop=True)

# Remover linhas sem os dois vetores principais
df = df.dropna(subset=['Forca_A', 'Forca_B'])

# Colunas derivadas
df['Media']    = (df['Forca_A'] + df['Forca_B']) / 2
df['Delta_AB'] = (df['Forca_A'] - df['Forca_B']).abs()

# Filtro IQR no Delta_AB (remove leituras discrepantes entre vetores)
Q1  = df['Delta_AB'].quantile(0.25)
Q3  = df['Delta_AB'].quantile(0.75)
IQR = Q3 - Q1
threshold = Q3 + IQR_MULTIPLIER * IQR

n_antes = len(df)
df = df[df['Delta_AB'] <= threshold].reset_index(drop=True)
n_depois = len(df)

print(f'Threshold Delta_AB : {threshold:.2f}')
print(f'Antes do filtro    : {n_antes:,} leituras')
print(f'Após o filtro      : {n_depois:,} leituras  (-{n_antes - n_depois})')
df[['Timestamp', 'Forca_A', 'Forca_B', 'Media', 'Delta_AB']].head()

Threshold Delta_AB : 111.45
Antes do filtro    : 2,429 leituras
Após o filtro      : 2,359 leituras  (-70)


,Timestamp,Forca_A,Forca_B,Media,Delta_AB
0,2022-06-27 15:16:07-03:00,712.2,689.40,700.800,22.80
1,2022-07-04 03:08:01-03:00,608.9,563.50,586.200,45.40
2,2022-07-05 03:03:01-03:00,742.9,730.75,736.825,12.15
3,2022-07-11 03:09:40-03:00,752.1,708.20,730.150,43.90
4,2022-07-12 19:49:01-03:00,375.6,350.50,363.050,25.10


## 4. target_rul — horas até a próxima troca confirmada

In [20]:
# Sincronizar troca_modulo.csv do SharePoint antes de carregar
from src.sp_troca_sync import sync_troca_modulo

try:
    sync_troca_modulo()
except Exception as _sync_err:
    print(f'⚠️  sp_troca_sync: {_sync_err}')
    print('   Usando arquivo local se disponível.')

# Carregar datas de troca
try:
    tc = pd.read_csv(ARQUIVO_TROCAS)
    troca_dates = pd.to_datetime(tc[COLUNA_DATA_TROCA], utc=True).sort_values().tolist()

    print(f'Trocas confirmadas carregadas: {len(troca_dates)}')
    print(f'  Primeira: {troca_dates[0].date()}')
    print(f'  Última  : {troca_dates[-1].date()}')

    # Calcular target_rul usando src/predictor.py
    df = build_rul_target(df, troca_dates=troca_dates)

    validos = df['target_rul'].notna().sum()
    print(f'\ntarget_rul calculado: {validos:,} leituras com valor  |  {len(df) - validos} NaN (pós-última troca)')
except FileNotFoundError:
    print('⚠️  troca_modulo.csv ausente — target_rul não será calculado.')
    print('   Isso é esperado para máquinas sem histórico de trocas no SharePoint.')
    troca_dates = []

Trocas confirmadas carregadas: 11
  Primeira: 2022-06-08
  Última  : 2025-11-24

target_rul calculado: 2,135 leituras com valor  |  224 NaN (pós-última troca)


## 5. Features de degradação

In [21]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    features = extract_degradation_features(df)

# Juntar features ao df principal
df = pd.concat([df.reset_index(drop=True), features.reset_index(drop=True)], axis=1)
df = df.loc[:, ~df.columns.duplicated()]

feature_cols = ['slope_Media_14d', 'std_Delta_AB_7d', 'media_ratio_14d']
print('Features calculadas:')
print(df[feature_cols].describe().T[['mean', 'std', 'min', 'max']].to_string())

Features calculadas:
                      mean        std          min          max
slope_Media_14d   2.037736  92.684862 -2862.511457  2443.726452
std_Delta_AB_7d  29.501974   7.236338     0.000000    60.616729
media_ratio_14d   1.012818   0.225195     0.397357     2.071007


## 6. Exportar hour_prev.csv

In [22]:
COLUNAS_EXPORT = [
    'Timestamp',
    'Forca_A', 'Forca_B',
    'Media', 'Delta_AB',
    'target_rul',
    'slope_Media_14d', 'std_Delta_AB_7d', 'media_ratio_14d',
]
cols_presentes = [c for c in COLUNAS_EXPORT if c in df.columns]
cols_ausentes  = [c for c in COLUNAS_EXPORT if c not in df.columns]

if cols_ausentes:
    print(f'⚠️  Colunas ausentes (serão omitidas): {cols_ausentes}')

df[cols_presentes].to_csv(OUTPUT_CSV, index=False)

print(f'Salvo: {OUTPUT_CSV}')
print(f'  Linhas  : {len(df):,}')
print(f'  Colunas : {cols_presentes}')
print(f'  Período : {df["Timestamp"].min()} → {df["Timestamp"].max()}')


Salvo: 00_hour_prev.csv
  Linhas  : 2,359
  Colunas : ['Timestamp', 'Forca_A', 'Forca_B', 'Media', 'Delta_AB', 'target_rul', 'slope_Media_14d', 'std_Delta_AB_7d', 'media_ratio_14d']
  Período : 2022-06-27 15:16:07-03:00 → 2026-04-29 20:04:45-03:00
